# NCA Workbench

Notebook для локального и серверного запуска baseline NCA.

- Данные на диске: `[T, H, W, F_data]`
- Состояние модели: `[B, data_channels + hidden_channels, H, W]`
- `primary_channel` используется для публичных метрик и визуализаций
- `loss_channels` управляет supervised loss: `primary` или `all_observed`
- Все импорты идут только через пакет `NCA`
- Все пути строятся только через `PROJECT_ROOT`


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "NCA" / "__init__.py").exists():
            return candidate
        if (candidate / "Real_game_of_life" / "NCA" / "__init__.py").exists():
            return candidate / "Real_game_of_life"
        if (candidate / "NCA" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing NCA/__init__.py")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd =", Path.cwd().resolve())
print("project_root =", PROJECT_ROOT)
print("sys.path[0:3] =", sys.path[:3])


cwd = D:\Proga\Game_of_life\Real_game_of_life\NCA
project_root = D:\Proga\Game_of_life\Real_game_of_life
sys.path[0:3] = ['D:\\Proga\\Game_of_life\\Real_game_of_life', 'd:\\Proga\\Game_of_life\\Real_game_of_life\\NCA', 'd:\\Anaconda3\\NewAnaconda\\python312.zip']


In [1]:
pip install torch_geometric

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 4.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import torch

from NCA.dataset import build_dataloaders, discover_npy_files
from NCA.model import NCA
from NCA.train import deterministic_eval, stochastic_eval, train_epoch
from NCA.utils import (
    build_initial_state,
    compute_binary_mask_metrics,
    get_device,
    load_checkpoint,
    rollout_model,
    save_checkpoint,
    set_seed,
    visible_to_probability,
)
from NCA.visualize import (
    plot_metric_curves,
    plot_triptych,
    plot_uncertainty_heatmap,
    save_rollout_animation,
)


In [1]:
pip install CUDA

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement CUDA (from versions: none)
ERROR: No matching distribution found for CUDA


In [ ]:
!git push --force

^C


remote: warning: File GNN/cache/frame_graphs_dynamic.pt is 58.65 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB        
remote: warning: File HeLa_Database/HeLa клетки/shape_division_analysis_dynamic/spot_shape_division_dataset.parquet is 95.64 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB        
remote: error: Trace: 71c52e2633274016eac8b80a35a367b0f01c328ee165f2128787130737e8b66c        
remote: error: See https://gh.io/lfs for more information.        
remote: error: File HeLa_Database/HeLa клетки/shape_division_analysis_output/spot_shape_division_dataset.parquet is 114.09 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: File HeLa_Database/6139958/6139958/20210904_TL2 - R05-C03-F0.tif is 667.75 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: File HeLa_Database/H2BmCherry_timelapse_60h/H2BmCherry_timelapse_60h/H2BmCherry_timelapse_60h.tif is 549.94 MB; this exceeds Git

In [3]:
CONFIG = {
    "project_root": PROJECT_ROOT,
    "data_root": PROJECT_ROOT / "NCA" / "data",
    "pattern": "*.npy",
    "run_dir": PROJECT_ROOT / "NCA" / "runs" / "workbench",
    "split_mode": "by_file",
    "split_ratios": (0.6, 0.2, 0.2),
    "train_steps": (8, 16),
    "eval_steps": {
        "one_step": 1,
        "rollout": 16,
        "stochastic": 16,
    },
    "batch_size": 8,
    "epochs": 5,
    "lr": 1e-3,
    "kernel_size": 3,
    "model_width": 64,
    "update_prob": 0.5,
    "data_channels": 2,
    "hidden_channels": 8,
    "primary_channel": 0,
    "loss_channels": "primary",
    "supervised_loss": "bce_dice",
    "bce_pos_weight": 6.0,
    "lambda_dice": 0.5,
    "eval_threshold": 0.5,
    "num_rollouts": 8,
    "loss_mode": "hybrid",
    "lambda_intermediate": 0.5,
    "lambda_hidden_l2": 1e-4,
    "use_alive_mask": False,
    "seed": 0,
}

CONFIG["data_root"].mkdir(parents=True, exist_ok=True)
CONFIG["run_dir"].mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()

print("device =", device)
print("data_root =", CONFIG["data_root"])
print("run_dir =", CONFIG["run_dir"])


device = cpu
data_root = D:\Proga\Game_of_life\Real_game_of_life\NCA\data
run_dir = D:\Proga\Game_of_life\Real_game_of_life\NCA\runs\workbench


In [4]:
files = discover_npy_files(CONFIG["data_root"], CONFIG["pattern"])
print(f"Found {len(files)} files")
sample = np.load(files[0])
print("Example shape:", sample.shape)
print("Expected disk format [T, H, W, F_data]")
print("Configured data_channels:", CONFIG["data_channels"])


Found 57 files
Example shape: (42, 64, 64, 2)
Expected disk format [T, H, W, F_data]
Configured data_channels: 2


In [5]:
loaders, normalizer = build_dataloaders(
    data_root=CONFIG["data_root"],
    pattern=CONFIG["pattern"],
    split_mode=CONFIG["split_mode"],
    split_ratios=CONFIG["split_ratios"],
    train_steps=CONFIG["train_steps"],
    eval_steps=CONFIG["eval_steps"],
    batch_size=CONFIG["batch_size"],
    eval_batch_size=CONFIG["batch_size"],
    seed=CONFIG["seed"],
    data_channels=CONFIG["data_channels"],
    primary_channel=CONFIG["primary_channel"],
)

train_batch = next(iter(loaders["train"]))
print("input_visible:", train_batch["input_visible"].shape)
print("targets_visible:", train_batch["targets_visible"].shape)
print("horizons:", train_batch["horizons"])


input_visible: torch.Size([8, 2, 64, 64])
targets_visible: torch.Size([8, 16, 2, 64, 64])
horizons: tensor([13, 11, 11, 12, 16,  9, 15,  9])


In [6]:
model = NCA(
    state_channels=CONFIG["data_channels"] + CONFIG["hidden_channels"],
    model_width=CONFIG["model_width"],
    kernel_size=CONFIG["kernel_size"],
    update_prob=CONFIG["update_prob"],
    use_alive_mask=CONFIG["use_alive_mask"],
    primary_channel=CONFIG["primary_channel"],
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
model


NCA(
  (conv1): Conv2d(10, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

In [ ]:
import base64

from IPython.display import HTML, Markdown, clear_output, display
from tqdm.auto import tqdm

history = []
epoch_bar = tqdm(range(CONFIG["epochs"]), desc="epochs")

for epoch in epoch_bar:
    train_metrics = train_epoch(
        model,
        loaders["train"],
        optimizer,
        device=device,
        loss_mode=CONFIG["loss_mode"],
        lambda_intermediate=CONFIG["lambda_intermediate"],
        lambda_hidden_l2=CONFIG["lambda_hidden_l2"],
        data_channels=CONFIG["data_channels"],
        hidden_channels=CONFIG["hidden_channels"],
        loss_channels=CONFIG["loss_channels"],
        primary_channel=CONFIG["primary_channel"],
        supervised_loss=CONFIG["supervised_loss"],
        bce_pos_weight=CONFIG["bce_pos_weight"],
        lambda_dice=CONFIG["lambda_dice"],
        show_progress=True,
        progress_desc=f"train epoch {epoch}",
    )

    val_det = deterministic_eval(
        model,
        loaders["val_rollout"],
        device=device,
        data_channels=CONFIG["data_channels"],
        hidden_channels=CONFIG["hidden_channels"],
        primary_channel=CONFIG["primary_channel"],
        supervised_loss=CONFIG["supervised_loss"],
        threshold=CONFIG["eval_threshold"],
        show_progress=True,
        progress_desc=f"det eval epoch {epoch}",
    )

    val_stoch = stochastic_eval(
        model,
        loaders["val_rollout"],
        device=device,
        num_rollouts=CONFIG["num_rollouts"],
        data_channels=CONFIG["data_channels"],
        hidden_channels=CONFIG["hidden_channels"],
        primary_channel=CONFIG["primary_channel"],
        supervised_loss=CONFIG["supervised_loss"],
        threshold=CONFIG["eval_threshold"],
        show_progress=True,
        progress_desc=f"stoch eval epoch {epoch}",
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_det_{k}": v for k, v in val_det.items()},
        **{f"val_stoch_{k}": v for k, v in val_stoch.items()},
    }
    history.append(row)

    epoch_bar.set_postfix(
        train_loss=f"{row['train_loss']:.3e}",
        det_roll=f"{row['val_det_rollout_mse']:.3e}",
        stoch_exp=f"{row['val_stoch_expected_mse']:.3e}",
    )

    clear_output(wait=True)
    display(Markdown(f"## Epoch {epoch + 1}/{CONFIG['epochs']}"))
    display(row)
    display(history[-5:])

save_checkpoint(
    CONFIG["run_dir"] / "checkpoint_latest.pt",
    model,
    optimizer,
    CONFIG["epochs"] - 1,
    {"config": CONFIG, "normalizer": normalizer.state_dict()},
    history[-1],
)

history[-1]


## Epoch 5/5

{'epoch': 4,
 'train_loss': 0.6555026963091733,
 'train_final_loss': 0.4177144724026061,
 'train_intermediate_loss': 0.4741839295939395,
 'train_hidden_penalty': 6.962575577853019,
 'val_det_one_step_mse': 0.018924035597592592,
 'val_det_rollout_mse': 0.02172508059690396,
 'val_det_population_mass_error': 0.7292588667737113,
 'val_det_dice': 0.7317336118883557,
 'val_det_iou': 0.5806921414203114,
 'val_det_precision': 0.5991322414742576,
 'val_det_recall': 0.9436731934547424,
 'val_stoch_expected_mse': 0.03219367522332403,
 'val_stoch_ensemble_mean_mse': 0.02528692300741871,
 'val_stoch_pixelwise_std_mean': 0.04438066285931402,
 'val_stoch_mass_std': 3.9311868614620633,
 'val_stoch_population_mass_error': 1.169472485780716,
 'val_stoch_dice': 0.7533462196588516,
 'val_stoch_iou': 0.6073760555850135,
 'val_stoch_precision': 0.6340627123912176,
 'val_stoch_recall': 0.9299582458204694}

[{'epoch': 0,
  'train_loss': 0.8964423647052363,
  'train_final_loss': 0.5397121176907891,
  'train_intermediate_loss': 0.7102810306507245,
  'train_hidden_penalty': 15.897338603582364,
  'val_det_one_step_mse': 0.13148613481058014,
  'val_det_rollout_mse': 0.031821762677282095,
  'val_det_population_mass_error': 1.1112928655412462,
  'val_det_dice': 0.7150795012712479,
  'val_det_iou': 0.5594447818067338,
  'val_det_precision': 0.576464037100474,
  'val_det_recall': 0.9448985242181354,
  'val_stoch_expected_mse': 0.04433595409823789,
  'val_stoch_ensemble_mean_mse': 0.038894160391969815,
  'val_stoch_pixelwise_std_mean': 0.05363343149009678,
  'val_stoch_mass_std': 4.419902192221747,
  'val_stoch_population_mass_error': 1.9771100680033367,
  'val_stoch_dice': 0.7318660186396705,
  'val_stoch_iou': 0.5792215259538757,
  'val_stoch_precision': 0.6019597732358508,
  'val_stoch_recall': 0.9351691040727828},
 {'epoch': 1,
  'train_loss': 0.7133075024997979,
  'train_final_loss': 0.4374840

{'epoch': 4,
 'train_loss': 0.6555026963091733,
 'train_final_loss': 0.4177144724026061,
 'train_intermediate_loss': 0.4741839295939395,
 'train_hidden_penalty': 6.962575577853019,
 'val_det_one_step_mse': 0.018924035597592592,
 'val_det_rollout_mse': 0.02172508059690396,
 'val_det_population_mass_error': 0.7292588667737113,
 'val_det_dice': 0.7317336118883557,
 'val_det_iou': 0.5806921414203114,
 'val_det_precision': 0.5991322414742576,
 'val_det_recall': 0.9436731934547424,
 'val_stoch_expected_mse': 0.03219367522332403,
 'val_stoch_ensemble_mean_mse': 0.02528692300741871,
 'val_stoch_pixelwise_std_mean': 0.04438066285931402,
 'val_stoch_mass_std': 3.9311868614620633,
 'val_stoch_population_mass_error': 1.169472485780716,
 'val_stoch_dice': 0.7533462196588516,
 'val_stoch_iou': 0.6073760555850135,
 'val_stoch_precision': 0.6340627123912176,
 'val_stoch_recall': 0.9299582458204694}

: 

In [ ]:
def _data_uri(path: Path) -> str:
    suffix = path.suffix.lower()
    mime = {
        ".png": "image/png",
        ".gif": "image/gif",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
    }.get(suffix, "application/octet-stream")
    payload = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{payload}"


loss_curve_path = plot_metric_curves(
    history,
    CONFIG["run_dir"] / "loss_curve.png",
    ["train_loss", "val_det_rollout_mse", "val_stoch_expected_mse"],
)
# display(HTML(
#     f"""
#     <div style='margin: 12px 0 8px;'>
#       <div style='font-size: 20px; font-weight: 700; margin-bottom: 10px;'>Training Curves</div>
#       <img src='{_data_uri(loss_curve_path)}' style='max-width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
#     </div>
#     """
# ))
loss_curve_path


In [ ]:
batch = next(iter(loaders["val_rollout"]))
visible = batch["input_visible"].to(device)
state0 = build_initial_state(
    visible,
    hidden_channels=CONFIG["hidden_channels"],
    hidden_init="zeros",
)

steps = int(batch["horizons"].max().item())
det_rollout = rollout_model(model, state0, steps=steps, stochastic=False)
stoch_rollouts = torch.stack(
    [
        rollout_model(model, state0, steps=steps, stochastic=True)
        for _ in range(CONFIG["num_rollouts"])
    ],
    dim=0,
)

triptych_path = plot_triptych(
    batch["input_visible"][0],
    batch["targets_visible"][0, -1],
    visible_to_probability(
        det_rollout[-1, 0, :CONFIG["data_channels"]].detach().cpu(),
        supervised_loss=CONFIG["supervised_loss"],
    ),
    CONFIG["run_dir"] / "triptych.png",
    title=f"Primary observed channel #{CONFIG['primary_channel']}",
    channel_index=CONFIG["primary_channel"],
)

prob_prediction = visible_to_probability(
    det_rollout[-1, 0, CONFIG["primary_channel"]:CONFIG["primary_channel"] + 1].detach().cpu(),
    supervised_loss=CONFIG["supervised_loss"],
)
target_primary = batch["targets_visible"][0, -1, CONFIG["primary_channel"]:CONFIG["primary_channel"] + 1].detach().cpu()
thresholded_prediction = (prob_prediction >= CONFIG["eval_threshold"]).float()
binary_metrics = compute_binary_mask_metrics(
    prob_prediction.unsqueeze(0),
    target_primary.unsqueeze(0),
    threshold=CONFIG["eval_threshold"],
)

threshold_triptych_path = plot_triptych(
    batch["input_visible"][0],
    target_primary,
    thresholded_prediction,
    CONFIG["run_dir"] / "triptych_thresholded.png",
    title=f"Thresholded prediction @ {CONFIG['eval_threshold']:.2f}",
    channel_index=CONFIG["primary_channel"],
)

uncertainty_path = plot_uncertainty_heatmap(
    stoch_rollouts.detach().cpu(),
    CONFIG["run_dir"] / "uncertainty.png",
    visible_channel=CONFIG["primary_channel"],
    step_index=-1,
)

animation_path = save_rollout_animation(
    visible_to_probability(
        det_rollout[:, 0, :CONFIG["data_channels"]].detach().cpu(),
        supervised_loss=CONFIG["supervised_loss"],
    ),
    CONFIG["run_dir"] / "det_rollout.gif",
    channel_index=CONFIG["primary_channel"],
)

# display(HTML(
#     f"""
#     <div style='margin: 12px 0 8px;'>
#       <div style='font-size: 20px; font-weight: 700; margin-bottom: 10px;'>Visual Diagnostics</div>
#       <div style='margin-bottom: 12px; font-size: 14px;'>Dice: {binary_metrics['dice']:.4f} | IoU: {binary_metrics['iou']:.4f} | Precision: {binary_metrics['precision']:.4f} | Recall: {binary_metrics['recall']:.4f}</div>
#       <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(320px, 1fr)); gap: 16px;'>
#         <div>
#           <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Raw Probability</div>
#           <img src='{_data_uri(triptych_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
#         </div>
#         <div>
#           <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Thresholded Mask</div>
#           <img src='{_data_uri(threshold_triptych_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
#         </div>
#         <div>
#           <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Uncertainty</div>
#           <img src='{_data_uri(uncertainty_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
#         </div>
#         <div>
#           <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Deterministic Rollout</div>
#           <img src='{_data_uri(animation_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
#         </div>
#       </div>
#     </div>
#     """
# ))

triptych_path, threshold_triptych_path, uncertainty_path, animation_path


## Baseline protocol

- `deterministic_eval(stochastic=False)` использовать как воспроизводимый benchmark.
- `stochastic_eval(stochastic=True, K rollouts)` использовать как вероятностную оценку динамики.
- Все публичные heatmap и основные метрики считаются только по `primary_channel`.
- По умолчанию сохраняется старый режим: `data_channels=1`, `hidden_channels=1`, `primary_channel=0`, `loss_channels='primary'`.
